# hw 4

configuration scripting & experiment runner

Andrew Chang-DeWitt \
CS 451
Spring 2026

---

automates the processes to execute the distributed version of Conway's Game of Life built for this assignment by

1. provisioning FABRIC resources
2. building main & worker binaries (production & debug for profiling) from source
3. executing test runs & recording results

first, we begin with

## Provision FABRIC resources

creates a slice of 11 nodes (10 workers to compute parts of each step + 1 main to coordinate the workers & log results).

In [2]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager(project_id="f5b4fc7d-978f-45be-9530-b38db8ef5046")

User: achangdewitt@hawk.illinoistech.edu bastion key is valid!
Configuration is valid


In [2]:
fablib.verify_and_configure()

User: achangdewitt@hawk.illinoistech.edu bastion keys do not exist or are expired.
Bastion Key saved at location: /home/fabric/work/fabric_config/fabric_bastion_key
Configuration is valid
Please save the config!


In [4]:
from fabrictestbed_extensions.fablib.node import Interface, Node

name = 'hw_04'
image = 'default_ubuntu_22'
site = "INDI"

In [ ]:
slice = fablib.new_slice(name = name)

all_nodes: list[Node] = []
nics: list[Interface] = []

def init_node(name: str) -> None:
    node = slice.add_node(name=name, image=image, cores=2, ram=4, disk=9, site=site)
    try:
        nic = node.add_component(model="NIC_Basic", name="iface1").get_interfaces()[0]  # type: ignore
    except KeyError:
        print("    expected nics to be list!")
        exit(1)
    all_nodes.append(node)
    nics.append(nic)

init_node("main")
for i in range(1, 11):
    init_node(f"worker{i}")
    
slice.add_l2network(name="net", interfaces=nics)
slice.submit()


Retry: 15, Time: 364 sec


ID,080298d8-89b2-431f-8c7b-2e5a1b1a9d82
Name,hw_04
Lease Expiration (UTC),2026-05-08 16:14:45 +0000
Lease Start (UTC),2026-05-07 16:14:45 +0000
Project ID,f5b4fc7d-978f-45be-9530-b38db8ef5046
State,StableOK
Email,achangdewitt@hawk.illinoistech.edu
UserId,c21c6e45-2dbc-4b02-bdf4-940eb7e9d1cf


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
e8432f75-1e7b-482a-ae38-11d3644d3b1c,main,2,4,10,default_ubuntu_22,qcow2,indi-w2.fabric-testbed.net,INDI,ubuntu,2001:18e8:fff0:3:f816:3eff:fefe:a659,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:18e8:fff0:3:f816:3eff:fefe:a659,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
e518eac0-1e28-47f0-bc23-8ff2b620fe2c,worker1,2,4,10,default_ubuntu_22,qcow2,indi-w2.fabric-testbed.net,INDI,ubuntu,2001:18e8:fff0:3:f816:3eff:fe67:a3e4,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:18e8:fff0:3:f816:3eff:fe67:a3e4,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
b6f40526-4510-4417-b299-bef37d849013,worker10,2,4,10,default_ubuntu_22,qcow2,indi-w2.fabric-testbed.net,INDI,ubuntu,2001:18e8:fff0:3:f816:3eff:feab:1407,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:18e8:fff0:3:f816:3eff:feab:1407,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
34dfa4f2-0e2b-410a-a594-ab4a419be836,worker2,2,4,10,default_ubuntu_22,qcow2,indi-w2.fabric-testbed.net,INDI,ubuntu,2001:18e8:fff0:3:f816:3eff:fed8:327a,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:18e8:fff0:3:f816:3eff:fed8:327a,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
7707d561-96b6-43ab-93c2-4c96d1258466,worker3,2,4,10,default_ubuntu_22,qcow2,indi-w2.fabric-testbed.net,INDI,ubuntu,2001:18e8:fff0:3:f816:3eff:fe92:dc77,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:18e8:fff0:3:f816:3eff:fe92:dc77,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
76be8e9f-1487-4502-931b-e83f641cfefc,worker4,2,4,10,default_ubuntu_22,qcow2,indi-w2.fabric-testbed.net,INDI,ubuntu,2001:18e8:fff0:3:f816:3eff:fe88:c06c,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:18e8:fff0:3:f816:3eff:fe88:c06c,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
3dab041d-a063-4e1e-9e48-bb3764a0a82b,worker5,2,4,10,default_ubuntu_22,qcow2,indi-w2.fabric-testbed.net,INDI,ubuntu,2001:18e8:fff0:3:f816:3eff:fe89:b6a3,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:18e8:fff0:3:f816:3eff:fe89:b6a3,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
89bd5ce6-485a-4525-8f09-3c463b99b933,worker6,2,4,10,default_ubuntu_22,qcow2,indi-w2.fabric-testbed.net,INDI,ubuntu,2001:18e8:fff0:3:f816:3eff:fe8c:a0a,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:18e8:fff0:3:f816:3eff:fe8c:a0a,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
53dd3db8-f2d0-4deb-be60-b25747a8efc0,worker7,2,4,10,default_ubuntu_22,qcow2,indi-w2.fabric-testbed.net,INDI,ubuntu,2001:18e8:fff0:3:f816:3eff:feb4:1d63,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:18e8:fff0:3:f816:3eff:feb4:1d63,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
7f43116d-9f43-4cd2-983e-41709856ebc5,worker8,2,4,10,default_ubuntu_22,qcow2,indi-w2.fabric-testbed.net,INDI,ubuntu,2001:18e8:fff0:3:f816:3eff:fefa:d5bb,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:18e8:fff0:3:f816:3eff:fefa:d5bb,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_

ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
36f00d3e-e03e-4288-8746-48a4bf23dd34,net,L2,L2Bridge,INDI,None,None,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
main-iface1-p1,p1,main,net,100,config,,06:48:7F:9F:8F:6F,enp7s0,enp7s0,fe80::448:7fff:fe9f:8f6f,4,HundredGigE0/0/0/9
worker1-iface1-p1,p1,worker1,net,100,config,,06:4D:C7:39:CA:17,enp7s0,enp7s0,fe80::44d:c7ff:fe39:ca17,4,HundredGigE0/0/0/9
worker2-iface1-p1,p1,worker2,net,100,config,,0A:C1:0A:FB:E8:C6,enp7s0,enp7s0,fe80::8c1:aff:fefb:e8c6,4,HundredGigE0/0/0/9
worker3-iface1-p1,p1,worker3,net,100,config,,06:CC:82:A2:A1:14,enp7s0,enp7s0,fe80::4cc:82ff:fea2:a114,4,HundredGigE0/0/0/9
worker4-iface1-p1,p1,worker4,net,100,config,,0E:20:23:22:44:FB,enp7s0,enp7s0,fe80::c20:23ff:fe22:44fb,4,HundredGigE0/0/0/9
worker5-iface1-p1,p1,worker5,net,100,config,,0E:F4:12:0D:95:77,enp7s0,enp7s0,fe80::cf4:12ff:fe0d:9577,4,HundredGigE0/0/0/9
worker6-iface1-p1,p1,worker6,net,100,config,,12:37:84:6B:C7:0E,enp7s0,enp7s0,fe80::1037:84ff:fe6b:c70e,4,HundredGigE0/0/0/9
worker7-iface1-p1,p1,worker7,net,100,config,,12:69:0E:4C:A1:84,enp7s0,enp7s0,fe80::1069:eff:fe4c:a184,4,HundredGigE0/0/0/9
worker8-iface1-p1,p1,worker8,net,100,config,,12:7C:74:9A:AC:D8,enp7s0,enp7s0,fe80::107c:74ff:fe9a:acd8,4,HundredGigE0/0/0/9
worker9-iface1-p1,p1,worker9,net,100,config,,12:FB:50:B5:B3:1A,enp7s0,enp7s0,fe80::10fb:50ff:feb5:b31a,4,HundredGigE0/0/0/9



Time to print interfaces 376 seconds


'080298d8-89b2-431f-8c7b-2e5a1b1a9d82'

In [5]:
slice = fablib.get_slice(name)
slice.show()

ID,080298d8-89b2-431f-8c7b-2e5a1b1a9d82
Name,hw_04
Lease Expiration (UTC),2026-05-08 16:14:45 +0000
Lease Start (UTC),2026-05-07 16:14:45 +0000
Project ID,f5b4fc7d-978f-45be-9530-b38db8ef5046
State,StableOK
Email,achangdewitt@hawk.illinoistech.edu
UserId,c21c6e45-2dbc-4b02-bdf4-940eb7e9d1cf


ID,080298d8-89b2-431f-8c7b-2e5a1b1a9d82
Name,hw_04
Lease Expiration (UTC),2026-05-08 16:14:45 +0000
Lease Start (UTC),2026-05-07 16:14:45 +0000
Project ID,f5b4fc7d-978f-45be-9530-b38db8ef5046
State,StableOK
Email,achangdewitt@hawk.illinoistech.edu
UserId,c21c6e45-2dbc-4b02-bdf4-940eb7e9d1cf


## Build binaries

first get paths to all nodes & configure their network interfaces.

In [ ]:
main_node = slice.get_node("main")
worker_nodes = [slice.get_node(f"worker{i}") for i in range(1, 11)]
all_nodes = [main_node] + worker_nodes

for i, node in enumerate(all_nodes):
    iface = node.get_interfaces()[0].get_os_interface()
    node.execute(f"sudo ip addr add 10.0.0.{i + 1}/24 dev {iface} && sudo ip link set {iface} up")

then install build dependencies & acquire source from github.

In [40]:
REPO_DIR = "~/hw_04"

install_th = [
    node.execute_thread(
        "sudo apt-get update -y && sudo apt-get install -y gcc make valgrind binutils"
    ) for node in all_nodes]
download_th = [
    node.execute_thread(
        f"git clone https://github.com/andrew-chang-dewitt/cs_451-hw_04-berkeley_sockets.git {REPO_DIR}"
    ) for node in all_nodes
]
for t in install_th + download_th:
    t.result()

set up directory to capture outputs for deliverables.

In [34]:
from pathlib import Path

REPORT_DIR = Path("report")
REPORT_DIR.mkdir(exist_ok=True)

def save(prefix: str, stdout: str, stderr: str) -> None:
    with (REPORT_DIR / f"{prefix}.out").open("a") as out:
        out.write(stdout)
    with (REPORT_DIR / f"{prefix}.err").open("a") as err:
        err.write(stderr)

finally, build a main binary on the main node & a worker binary on each worker node.

In [36]:
worker_build_threads = [
    node.execute_thread(f"cd {REPO_DIR} && make clean && make worker")
    for node in worker_nodes
]

main_build_out, main_build_err = main_node.execute(
    f"cd {REPO_DIR} && make clean && make berk"
)
save("a", main_build_out, main_build_err)

worker_build_out, worker_build_err = worker_build_threads[0].result()
save("a", worker_build_out, worker_build_err)

for t in worker_build_threads[1:]:
    # this is actually a Future<Thread>, type signature on fablib.Node.execute_thread() is wrong
    t.result()  # type: ignore

rm -rf /home/ubuntu/hw_04/target
compiling dependency...
gcc -MT /home/ubuntu/hw_04/target/release/obj/args.o -MMD -MP -MF /home/ubuntu/hw_04/target/release/dep/args.d -Wall -Wextra -Wformat=2 -Wswitch-default -Wcast-align -Wpointer-arith -Wbad-function-cast -Wstrict-prototypes -Winline -Wundef -Wnested-externs -Wcast-qual -Wshadow -Wwrite-strings -Wconversion -Wunreachable-code -Wstrict-aliasing=2 -fno-common -fstrict-aliasing -std=c99 -pedantic -O3 -c /home/ubuntu/hw_04/args.c -o /home/ubuntu/hw_04/target/release/obj/args.o
...dependency /home/ubuntu/hw_04/target/release/obj/args.o built.
compiling dependency...
gcc -MT /home/ubuntu/hw_04/target/release/obj/peer.o -MMD -MP -MF /home/ubuntu/hw_04/target/release/dep/peer.d -Wall -Wextra -Wformat=2 -Wswitch-default -Wcast-align -Wpointer-arith -Wbad-function-cast -Wstrict-prototypes -Winline -Wundef -Wnested-externs -Wcast-qual -Wshadow -Wwrite-strings -Wconversion -Wunreachable-code -Wstrict-aliasing=2 -fno-common -fstrict-aliasing -st

## Execute test runs

In [ ]:
SIZE = 1581  # sqrt(50^2 * 1000) -> 1000x more cells than assignment 3
CYCLES = 100
NUM_WORKERS = 10
BASE_PORT = 9000
MAIN_IP = "10.0.0.1"

def make_init(size: int) -> str:
    rows = [["0"] * size for _ in range(3)]
    # glider, left side (rows 0-2, cols 1-4)
    rows[0][1] = "1"
    rows[1][2] = "1"
    rows[2][2] = "1"
    rows[2][3] = "1"
    rows[2][4] = "1"
    # horizontal blinker, right side (row 1, cols size-6 to size-4)
    b = size - 6
    rows[1][b] = "1"
    rows[1][b + 1] = "1"
    rows[1][b + 2] = "1"
    return "".join("".join(r) for r in rows)


INIT = make_init(SIZE)

main_run_th = main_node.execute_thread(
    f"cd {REPO_DIR} && time ./target/release/bin/berkeley_life"
    f" -s {SIZE} -c {CYCLES} -g {NUM_WORKERS} -P {BASE_PORT} -i {INIT}"
)
worker_run_threads = [
    node.execute_thread(
        f"cd {REPO_DIR} && ./target/release/bin/berkeley_worker"
        f" {MAIN_IP} {BASE_PORT + i}"
    )
    for i, node in enumerate(worker_nodes)
]

main_run_out, main_run_err = main_run_th.result()
save("b", main_run_out, main_run_err)

for t in worker_build_threads[1:]:
    # this is actually a Future<Thread>, type signature on fablib.Node.execute_thread() is wrong
    worker_run_out, worker_run_err = t.result()  # type: ignore
    save("b", worker_run_out, worker_run_err)

In [6]:
# Clean-up
fablib.delete_slice(name)